# Cryptography (CC4017) -- Week 3 [Extra]

## Q1: Weak Security

Unpredictability of key generation is a central requirement to the security of an encryption scheme.
If the key can be efficiently guessed, then no encryption scheme can ever be shown to be IND-CPA
secure, as any adversary can simply enumerate the possible keys and test for decryptions.
The code ciphersuite_aesnotrand.py is encrypting a block message using a very weak key. Check
it out to understand what it is doing wrong.

### Question - P1: 
Program q1.py produces weak_ciphertexts. Suppose you know that the encryptedmessage was “Attack at Dawn!!’ ’. Extend that program to read the file and guess the key used for that encryption

In [43]:
import ciphersuite_aesnotrand as ciphersuite
from binascii import hexlify, unhexlify

offset = 3
key = ciphersuite.gen()
msg = 'Attack at dawn!!'
cph = ciphersuite.enc(key, bytearray(msg,'ascii'))
print(key)

f = open("weak_ciphertext", "wb")
f.write(cph)
f.close()
## 
# Extend me to
# 1 - Read ciphertext
# 2 - Guess the key used
# 3 - Test the decryption
##

with open("weak_ciphertext", "rb") as f:
    cph_txt = f.read()

i = 0
while (i < 2**(offset*8)):
    curr_key = b'\x00' * (16 - offset) + i.to_bytes(offset, 'big')
    print(curr_key)
    i += 1 
    try:
        dcph = ciphersuite.dec(curr_key, cph_txt)
        if dcph.decode('ascii') == msg:
            print("Success!")
            print(f"Key found: {curr_key}")
            break
    except Exception as e:
        pass


b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x98'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x03'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x04'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x05'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x06'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x07'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x08'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\t'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\n'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x0b'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x0c'
b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x

### Question - P2: 
Increase the size of the offset in the ciphersuite. How large must it be for your
machine to be unable to test it in 3 hours?

Lets start by taking in account the time for one attempt at finding the key. Assuming each try takes around 1 ms , as can be tested by using the time library in python, being the decryption operation the most time consuming and the others near negligeble, we can say that in 3 hours, which are 10,800 seconds, we can attempt up to 10,800,000 different keys. Having in account the offset, we can say that the number of possible keys is given by 2 ^ (offset * 2 ). Using the inequality 2 ^ (offset * 2 ) > 10,800,000, that translates to an offset that takes longer than 3h, we get offset×8>log⁡2(108,000,000), and then offset>26.7/8​≈3.34. This means that an offset any higher than 3.34 bytes is going to take longer than 3h. The next possible offset is 4 bytes, since we are picking a random int in the ciphersuit code, meaning that when generating a key with up to 4 bytes of size we might take longer than 3h to find it 

## Q2: Fixed Initialization Vectors

Suppose our encryption scheme is AES-CBC using a fixed IV. Construct an attack against
the IND-CPA security experiment of this scheme, i.e. write an algorithm for our adversary to beat the
IND-CPA security experiment, namely:

- What are the queries performed to the encryption oracle
- What are the messages produced as m0 , m1
- How b is decided

Supposing the IV is fixed, that is its always the same IV every time we go to encrypt a message. Following the IND-CPA security experiment we known that the attacker will produce two messages of the same size and send them, awaiting the result of encryption of one of them, after that, he needs to pick which message was encrypted and wins if he is right. In order to prove that there is an attack that works against the scheme, we need to show that the attacker will have a probability of over 1/2 when guessing which plain text was cyphered. 
In this attack this adversary selects two different messages M0 and M1, both the size of a CBC block. Due to the deterministic nature of the IV being fixed, and the key used being the same during the attack, the adversary queries m0 and obtains C0. During the attack, he simply sends both messages, if Cb is equal to C0, then b equals 0, if not then b equals 1. In case he can't repeat messages all he has to do is send M0 + M0 and M1 or eventually M1+M0, since M0 is the size of a block and with the constant IV and key the results should be the same by just looking at the first part of the ciphered text.

## Q3: Predictable Initialization Vectors


Consider the following encryption scheme:
- Use the block encryption function (with the same key) on the nonce to generate an IV ← E(k, n)
- Compute the encryption of the message using AES-CBC with that IV


Observe that this prevents trivial attacks, such as setting the IV to 0 – as it is encrypted – and also
disallows fixing the IV – as the same nonce cannot be reused. However, the IV is predictable, and
that can lead to an attack.

### Question - P1: 
Construct an attack against the nonce-based IND-CPA security experiment of this scheme



Supposing this time that IV isn't fixed but is instead predictable, which means we can know/predict what the next IV is going to be. Following the IND-CPA security experiment we known that the attacker will produce two messages of the same size and send them, awaiting the result of encryption of one of them, after that, he needs to pick which message was encrypted and wins if he is right. In order to prove that there is an attack that works against the scheme, we need to show that the attacker will have a probability of over 1/2 when guessing which plain text was cyphered. 
Assuming the key stays fixed during the attack, and that an attacker has access or can predict the IV before the next query, all the attacker has to do initally is send M0. Having both messages M0 and M1, the size of the CBC block, the attacker simply has to send M0' as M0(the one sent previously) XOR previous IV(IV1) XOR current IV(IV2), and M1. If the Cb' generated is equal to C0 generated on the first attacker's query, then we know b = 0, other wise b = 1. This happens because, when preparing the message for encryption M0' XOR with IV2 happens, which translates to M0 XOR IV1 XOR IV2 XOR IV2, which becomes M0 XOR IV1, which was the message sent before.

### Question - P2:
 Write a program that prints the messages/ciphertexts used in this attack, and that
shows this IND-breaking correlation

In [55]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
from binascii import hexlify
import random

def xor_bytes(b1, b2):
    return bytes([x ^ y for x, y in zip(b1, b2)])

key = get_random_bytes(16)  
iv1 = get_random_bytes(16)  # IV for M0 encryption
iv2 = bytes([(iv1[i] + 1) % 256 for i in range(16)]) # Predictable next IV, used in the guessing game

m0 = b"Attack at dawn!!" #initially sent M0

m1 = b"Attack at dusk!!" # message for the guessing game
m0_ = xor_bytes(m0,xor_bytes(iv1, iv2)) # prepared message for the game the M0'

cph = AES.new(key, AES.MODE_CBC, iv=iv1) # first query for the attack
c0 = cph.encrypt(m0)

message = random.choice([m0_, m1])
oracl_cph = AES.new(key,AES.MODE_CBC, iv=iv2)
cb = oracl_cph.encrypt(message)

if(c0 == cb):
    print('Oracle chose 0')
else:
    print('Oracle chose 1')




Oracle chose 1


## Q4: Padding Attacks

Let k denote the next multiple of b for the message m. PKCS#7 padding entails filling the last k − |M |
bytes with value k − |M |, e.g.
• 0x01 means 1 byte of padding added with this value
• 0x03 means 3 bytes of padding added with this value

### Question - P1:

Consider a message that is already of size multiple of b. Why is it necessary to add
padding?

In this case, the padding is used in order to maintain a certain consistency. Padding is always added to the last block in order to equal the length of the message to a multiple of the block size. This means that the last block always has padding. In order to avoid confusion and to make sure the full encrypted message was received, there is added a block of padding, in order to denote that the algorithm in fact received the last block. This adding another block of padding is to tell that the transmission of blocks has reached the end. In other words, to avoid ambiguity, padding is always added.

### Question - P2:
Consider an AES-CBC encryption scheme that, upon decryption, produces an error
whenever a padding error occurs, i.e. if the decrypted message does not follow PKCS#7 padding.
How can an adversary that is given a ciphertext use a decryption oracle to extract information about
the original message?

When its decryption time, CBC mode will first de encrypt a give cypher text with the key and following that will XOR the obtained message with the previous ciphertext. Due to the oracle producing errors if the padding submitted is wrong, an attacker can now, by manipulating the previous cypher text, obtain the contents of the last plain text. In other words, in a message with two plain text blocks, P0 and P1, that are cyphered using AES-CBC following PKCS#7 padding, by manipulating C0, the attacker can obtain P1 without knowing the key. In order for the padding to be considered correct, each padding bytes needs to have the value corresponding to the number of padding bytes. Knowing this information an attacker can change C0 in order to obtain the temporary messgae block created after key decryption. The attack goes as following: in order to get a correct padding, on the least significant byte, there should 0x01,meaning 1 byte padding. By changing the value of the last byte, iterating over every possibility, we will alter what the plaintext will translate to. If the value is incorrect for the expected padding, the oracle will throw an error, meaning the attacker can keep going and changing the value until it no longer produces an error, meaning it successfully produced a plain text ending in 0x01. By doing 0x01 XOR the value of the manipulated byte, you can obtain the last byte of the temporary message created by key decryption. All the attacker needs to do next is find the value of the second byte, changing the first byte to produce a 0x02 when XOR'ing, and changing the value of the second byte of the cypher text until it gets the correct padding of 0x02/0x02. All the attacker has to do now is keep going for the rest of the 16 bytes of C0, in order to later obtain the temporary message created by decryption of P1 and XOR'ing it with C0 in order to obtain actual P1.